<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/First%20Test%20WA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 1: CLEAN DATA**

# Import Extensions And Open File

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
from collections import defaultdict
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

# Remove Unnecessary Columns

In [2]:
# List of columns to remove
columns_to_remove = [

    # Identifier columns
    'Customer Id',
    'Category Id',
    'Department Id',

    # Personal/customer detail columns
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Customer Full Name',

    # High-cardinality text columns
    'Product Description',
    'Product Image',
    'Product Name',


    # Duplicate or Redundant Geographic Columns

'Customer Segment',
'Order City',
'Order State',]

# Remove only columns that actually exist in dataframe
existing_columns_to_remove = [
    col for col in columns_to_remove if col in df.columns]

# Drop columns
df = df.drop(columns=existing_columns_to_remove)

# Display remaining columns
print("Remaining Columns:")
print(df.columns.tolist())

# Display shape after removal
print("\nNew Dataset Shape:")
print(df.shape)

Remaining Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']

New Dataset Shape:
(180519, 38)


# Identify missing or erroneous data


In [3]:

print(df.shape)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

(180519, 38)
Order Zipcode    155679
dtype: int64


In [4]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [5]:
missing = df.isnull().sum()
print(missing[missing > 0])

Order Zipcode    155679
dtype: int64


In [6]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
    cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [7]:
non_negative_columns = [
    'Sales',
    'Order Item Quantity',
    'Order Item Total',
    'Product Price',
    'Days for shipping (scheduled)']

for col in non_negative_columns:

    if col in df.columns:

        # Count invalid negatives
        invalid_count = (df[col] < 0).sum()

        print(f"\n{col} Negative Values: {invalid_count}")

        # Replace negatives with median
        median_value = df[df[col] >= 0][col].median()

        df.loc[df[col] < 0, col] = median_value


Sales Negative Values: 0

Order Item Quantity Negative Values: 0

Order Item Total Negative Values: 0

Product Price Negative Values: 0


In [8]:
numeric_cols = df.select_dtypes(include=np.number).columns

outlier_summary = {}

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary[col] = outliers

outlier_df = pd.DataFrame.from_dict(
    outlier_summary,
    orient='index',
    columns=['Outlier Count']
)

print(outlier_df.sort_values('Outlier Count', ascending=False))

                               Outlier Count
Order Zipcode                          24818
Benefit per order                      18942
Order Profit Per Order                 18942
Order Item Profit Ratio                17300
Order Item Discount                     7537
Order Item Product Price                2048
Product Price                           2048
Sales per customer                      1943
Order Item Total                        1943
Longitude                               1414
Order Customer Id                       1198
Sales                                    488
Latitude                                   9
Late_delivery_risk                         0
Days for shipment (scheduled)              0
Days for shipping (real)                   0
Order Item Quantity                        0
Order Item Id                              0
Order Item Cardprod Id                     0
Order Item Discount Rate                   0
Order Id                                   0
Product Ca

# Winsorize Outliers

In [9]:
for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

# Log Transform Highly Skewed Variables

In [10]:
skew_cols = [
    'Order Item Total',
    'Order Profit Per Order',
    'Shipping Cost',
    'Profit Per Unit'
]

for col in skew_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col] - df[col].min())

In [11]:
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nDataset Preview:")
print(df.head())


Remaining Missing Values:
0

Final Dataset Shape:
(180519, 38)

Dataset Preview:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                       3.0                            4.0   
1  TRANSFER                       5.0                            4.0   
2      CASH                       4.0                            4.0   
3     DEBIT                       3.0                            4.0   
4   PAYMENT                       2.0                            4.0   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1         -79.700005          311.359985     Late delivery   
2         -79.700005          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk   Category Name Customer City Customer Country  ...  \
0                 0.0  Sporting Goo

# Remove Constant and Near-Constant Features

In [12]:
constant_cols = [
    col for col in df.columns
    if df[col].nunique() == 1
]

print("Constant Features:")
print(constant_cols)

df.drop(columns=constant_cols, inplace=True)
near_constant_cols = []

for col in df.columns:

    freq = df[col].value_counts(normalize=True)

    # Check if freq is empty (e.g., column was entirely NaN) or if a single value dominates
    if freq.empty or freq.iloc[0] > 0.99:
        near_constant_cols.append(col)

print("Near Constant Features:")
print(near_constant_cols)

df.drop(columns=near_constant_cols, inplace=True)

Constant Features:
['Order Zipcode', 'Product Status']
Near Constant Features:
[]


# Remove Duplicate Rows

In [13]:
print("Rows before:", len(df))

df.drop_duplicates(inplace=True)

print("Rows after:", len(df))

Rows before: 180519
Rows after: 180519


# Verify Dataset Quality

In [14]:
print("Remaining Missing Values:")
print(df.isnull().sum().sum())

print("Dataset Shape:")
print(df.shape)

Remaining Missing Values:
0
Dataset Shape:
(180519, 36)


#Save Data

In [15]:
# Save cleaned RL dataset to CSV in Google Colab

output_file = 'dataco_rl_cleaned.csv'

# Save dataframe
df.to_csv(output_file, index=False)

print(f"Dataset saved as: {output_file}")

Dataset saved as: dataco_rl_cleaned.csv


# **STEP 2: FEATURE ENGINEERING**

In [16]:
df = pd.read_csv('dataco_rl_cleaned.csv', encoding='latin1')

print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 36)


# Date and Time Features

In [17]:
# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce')

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)


# Create a Product-Region Matrix and Network

In [18]:
# Create a Product-Region Matrix
product_region = (
    df.groupby("Product Category Id")["Order Region"]
      .nunique()
      .reset_index(name="num_regions")
      .sort_values("num_regions", ascending=False))

print(product_region.head(20))

    Product Category Id  num_regions
6                   9.0           23
4                   6.0           23
3                   5.0           23
16                 29.0           23
15                 26.0           23
13                 18.0           23
10                 13.0           23
9                  12.0           23
14                 24.0           23
12                 17.0           23
31                 46.0           23
27                 41.0           23
28                 43.0           23
29                 44.0           23
26                 40.0           23
23                 36.0           23
24                 37.0           23
22                 35.0           23
19                 32.0           23
30                 45.0           23


In [19]:
order_features = (
    df.groupby("Order Id")
      .agg(
          unique_products=("Product Category Id", "nunique"),
          total_quantity=("Order Item Quantity", "sum"),
          total_sales=("Sales", "sum"))
      .reset_index())

order_features["complex_order"] = (
    order_features["unique_products"] >= 3
).astype(int)

In [20]:
network = (
    df.groupby(["Product Category Id", "Order Region"])
      .agg(
          orders=("Order Id", "nunique"),
          quantity=("Order Item Quantity", "sum"),
          revenue=("Sales", "sum"))
      .reset_index())

# Explore Product Statistics

In [21]:
product_stats = (
    network.groupby("Product Category Id")
           .agg(
               regions_served=("Order Region", "nunique"),
               total_orders=("orders", "sum"),
               total_quantity=("quantity", "sum"),
               total_revenue=("revenue", "sum"))
           .reset_index())


# Define Multi-Region Stock

In [22]:
region_threshold = product_stats["regions_served"].median()
order_threshold = product_stats["total_orders"].median()

product_stats["likely_multi_region_stocked"] = (
    (product_stats["regions_served"] >= region_threshold)
    &
    (product_stats["total_orders"] >= order_threshold)
).astype(int)

#Create a stock score

In [23]:
product_stats["stocking_score"] = (
    0.5 *
    (
        product_stats["regions_served"]
        / product_stats["regions_served"].max())
    +
    0.5 *
    (
        product_stats["total_orders"]
        / product_stats["total_orders"].max()))


# Merge the network

In [24]:
network = network.merge(
    product_stats[
        [
            "Product Category Id",
            "regions_served",
            "stocking_score",
            "likely_multi_region_stocked"
        ]
    ],
    on="Product Category Id",
    how="left")

# Find Edge Weight

In [25]:
network["edge_weight"] = (
    0.4 *
    (network["orders"] / network["orders"].max())
    +
    0.3 *
    (network["quantity"] / network["quantity"].max())
    +
    0.3 *
    (network["revenue"] / network["revenue"].max()))


 # Find candidate shipping regions

In [26]:
candidate_regions = (
    network.sort_values(
        ["Product Category Id", "edge_weight"],
        ascending=[True, False])
    .groupby("Product Category Id")
    .head(5))

# Save Outputs

In [27]:
network.to_csv(
    "product_region_network.csv",
    index=False
)

candidate_regions.to_csv(
    "candidate_fulfillment_regions.csv",
    index=False
)

product_stats.to_csv(
    "product_stocking_scores.csv",
    index=False
)

print("Files created successfully")

Files created successfully


# Shipping Engineering

In [28]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'],
    errors='coerce'
)

# Actual shipping time

df['actual_ship_days'] = (
    df['shipping date (DateOrders)']
    - df['order date (DateOrders)']
).dt.days

# Difference from scheduled

df['shipping_delay'] = (
    df['Days for shipping (real)']
    - df['Days for shipment (scheduled)']
)

# Binary on-time flag

df['on_time_delivery'] = (
    df['shipping_delay'] <= 0
).astype(int)

# Late shipment flag

df['late_delivery'] = (
    df['shipping_delay'] > 0
).astype(int)

# Profit Engineering

In [29]:
# Margin percentage

df['margin_pct'] = (
    df['Order Profit Per Order']
    / df['Sales']
)

# Profit per item

df['profit_per_item'] = (
    df['Order Profit Per Order']
    / df['Order Item Quantity']
)

# Discount percentage

df['discount_pct'] = (
    df['Order Item Discount']
    / df['Order Item Product Price']
)

# Product Demand Features

In [30]:
product_stats = (
    df.groupby('Product Category Id')
      .agg(
          product_orders=('Order Id','nunique'),
          product_quantity=('Order Item Quantity','sum'),
          product_revenue=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    product_stats,
    on='Product Category Id',
    how='left'
)

# Regional Features

In [31]:
region_stats = (
    df.groupby('Order Region')
      .agg(
          region_orders=('Order Id','nunique'),
          region_sales=('Sales','sum'),
          region_profit=('Order Profit Per Order','sum')
      )
      .reset_index()
)

df = df.merge(
    region_stats,
    on='Order Region',
    how='left'
)

# Fulfillment Network Features

In [32]:
stocking = pd.read_csv(
    "product_stocking_scores.csv"
)

df = df.merge(
    stocking[
        [
            'Product Category Id',
            'stocking_score',
            'regions_served',
            'likely_multi_region_stocked'
        ]
    ],
    on='Product Category Id',
    how='left'
)

# Shipping Mode Features

In [33]:
speed_mapping = {
    'Same Day':4,
    'First Class':3,
    'Second Class':2,
    'Standard Class':1
}
df['shipping_mode_speed'] = df['Shipping Mode'].map(speed_mapping)

action_id_mapping = {
    'Standard Class': 0,    # Maps to key 0 in shipping_cost_map
    'Second Class': 1,     # Maps to key 1 in shipping_cost_map
    'First Class': 2,      # Maps to key 2 in shipping_cost_map
    'Same Day': 3          # Maps to key 3 in shipping_cost_map
}
df['action_id'] = df['Shipping Mode'].map(action_id_mapping)

shipping_cost_map = {
    0: 1.0,    # Standard
    1: 1.5,    # Second
    2: 2.0,    # First
    3: 3.0     # Same Day
}

BASE_COST = 10

df['Mode_Cost'] = (
    BASE_COST *
    df['action_id'].map(shipping_cost_map)
)

# Order Complexity Features

In [34]:
order_complexity = (
    df.groupby('Order Id')
      .agg(
          unique_products=('Product Category Id','nunique'),
          total_quantity=('Order Item Quantity','sum'),
          total_sales=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    order_complexity,
    on='Order Id',
    how='left'
)

df['complex_order'] = (
    df['unique_products'] >= 3
).astype(int)

# RL State Features

In [35]:
state_features = [
    'Product Category Id',
    'Order Region',
    'Market',
    'Shipping Mode',
    'stocking_score',
    'regions_served',
    'likely_multi_region_stocked',
    'unique_products',
    'total_quantity',
    'shipping_delay',
    'margin_pct',
    'profit_per_item',
    'order_month',
    'order_weekday'
]

# Reward Features

In [36]:
df['reward'] = (
    df['Order Profit Per Order']
    + 100 * df['on_time_delivery']
    - 20 * df['late_delivery']
)
LATE_PENALTY = 10

df['reward'] = (
    df['Order Profit Per Order']
    - df['Mode_Cost']
    - LATE_PENALTY * df['Late_delivery_risk']
)

# Save Engineered Dataset

In [37]:
df.to_csv(
    "dataco_rl_feature_engineered.csv",
    index=False
)

print(df.shape)

(180519, 65)


# **STEP 3: ENCODING CATEGORICAL VARIABLES**

In [38]:

df = pd.read_csv('dataco_rl_feature_engineered.csv', encoding='latin1')


print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 65)


In [39]:
one_hot_cols = [
    'Shipping Mode',
    'Market',
    'Order Region'
]

df = pd.get_dummies(
    df,
    columns=one_hot_cols,
    drop_first=False
)

df.to_csv(
    "dataco_rl_onehot.csv",
    index=False
)

print(df.shape)

(180519, 94)


# Filter on West Africa

In [42]:
df = pd.read_csv("dataco_rl_onehot.csv")

# Keep only West Africa orders
west_africa_df = df[df["Order Region_West Africa"] == 1].copy()

print("Number of West Africa orders:", len(west_africa_df))

Number of West Africa orders: 3696


# **STEP 4: TRAIN/TEST TEMPORAL SPLIT**

In [44]:
import pandas as pd

df = pd.read_csv("dataco_rl_onehot.csv")

df = df[df["Order Region_West Africa"] == 1].copy()

df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)']
)

df = df.sort_values(
    'order date (DateOrders)'
)

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(2587, 94)
(554, 94)
(555, 94)


In [45]:
# Save datasets

train_df.to_csv(
    "dataco_rl_train.csv",
    index=False
)

val_df.to_csv(
    "dataco_rl_validation.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test.csv",
    index=False
)

print("Files saved successfully")

Files saved successfully


# Recompute Numeric Columns

In [46]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

print(f"Number of numeric features: {len(numeric_cols)}")
print(numeric_cols)

Number of numeric features: 51
['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Late_delivery_risk', 'Latitude', 'Longitude', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Card Id', 'Product Category Id', 'Product Price', 'order_year', 'order_month', 'order_day', 'order_dayofweek', 'is_weekend', 'actual_ship_days', 'shipping_delay', 'on_time_delivery', 'late_delivery', 'margin_pct', 'profit_per_item', 'discount_pct', 'product_orders', 'product_quantity', 'product_revenue', 'region_orders', 'region_sales', 'region_profit', 'stocking_score', 'regions_served', 'likely_multi_region_stocked', 'shipping_mode_speed', 'action_id', 'Mode_Cost', 'unique_products', 'total_quantity', 'total_sales', 'complex_order', 'reward

# STEP 5: STATE-SPACE DISCRETIZATION

# Feature Selection

In [47]:
state_features = [
    'Profit Margin',
    'Shipping Cost Ratio',
    'Historical On-Time Rate',
    'Regional Delay Score',
    'Product Category Id'
]

# Fit Bins Using Training Data Only

In [48]:
# Load training data
train_df = pd.read_csv("dataco_rl_train.csv")

exclude_cols = [
    'Reward',
    'Late_delivery_risk',
    'Action_ID'
]

continuous_features = [
    col for col in train_df.select_dtypes(include='number').columns
    if col not in exclude_cols
]

bin_edges = {}

# Create 10 bins for each feature
for col in continuous_features:

    _, bins = pd.qcut(
        train_df[col],
        q=10,
        labels=False,
        retbins=True,
        duplicates='drop'
    )

    bin_edges[col] = bins

# Save bin definitions
joblib.dump(
    bin_edges,
    "state_bins.pkl"
)

print("Bins created successfully")

Bins created successfully


In [49]:
bin_counts = {}

for col in continuous_features:
    _, bins = pd.qcut(
        train_df[col],
        q=10,
        retbins=True,
        duplicates='drop'
    )

    bin_counts[col] = len(bins) - 1

print(bin_counts)

{'Days for shipping (real)': 5, 'Days for shipment (scheduled)': 3, 'Benefit per order': 10, 'Sales per customer': 10, 'Latitude': 10, 'Longitude': 10, 'Order Customer Id': 10, 'Order Id': 10, 'Order Item Cardprod Id': 9, 'Order Item Discount': 10, 'Order Item Discount Rate': 10, 'Order Item Id': 10, 'Order Item Product Price': 8, 'Order Item Profit Ratio': 10, 'Order Item Quantity': 4, 'Sales': 9, 'Order Item Total': 10, 'Order Profit Per Order': 10, 'Product Card Id': 9, 'Product Category Id': 9, 'Product Price': 8, 'order_year': 0, 'order_month': 4, 'order_day': 10, 'order_dayofweek': 6, 'is_weekend': 1, 'actual_ship_days': 5, 'shipping_delay': 5, 'on_time_delivery': 1, 'late_delivery': 1, 'margin_pct': 10, 'profit_per_item': 10, 'discount_pct': 10, 'product_orders': 8, 'product_quantity': 9, 'product_revenue': 8, 'region_orders': 0, 'region_sales': 0, 'region_profit': 0, 'stocking_score': 8, 'regions_served': 1, 'likely_multi_region_stocked': 1, 'shipping_mode_speed': 3, 'action_id

# Apply Bins to Train/Test Data

In [50]:
train_df = pd.read_csv("dataco_rl_train.csv")
test_df = pd.read_csv("dataco_rl_test.csv")

bin_edges = joblib.load("state_bins.pkl")

continuous_features = list(bin_edges.keys())

for col in continuous_features:

    train_df[col + "_bin"] = np.digitize(
        train_df[col],
        bin_edges[col][1:-1]
    )

    test_df[col + "_bin"] = np.digitize(
        test_df[col],
        bin_edges[col][1:-1]
    )

train_df.to_csv(
    "dataco_rl_train_discrete.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test_discrete.csv",
    index=False
)

print("Discretization complete")

Discretization complete


# Build RL States

In [51]:
# Redefine state_columns with existing features in train_df
state_columns = [
    'Product Category Id',
    'Category Name',
    'shipping_mode_speed',
    'stocking_score_bin',
    'margin_pct_bin',
    'shipping_delay_bin',
    'profit_per_item_bin',
    'unique_products_bin'
]

train_df['state'] = train_df[state_columns] \
    .astype(str) \
    .agg('|'.join, axis=1)

# Create State IDs

In [52]:
state_encoder = LabelEncoder()

train_df['state_id'] = state_encoder.fit_transform(
    train_df['state']
)

joblib.dump(
    state_encoder,
    "state_encoder.pkl"
)

print(
    "Unique states:",
    train_df['state_id'].nunique()
)

Unique states: 1650


# Create Action IDs

In [53]:
candidate_regions = pd.read_csv(
    "candidate_fulfillment_regions.csv"
)

shipping_modes = [
    "Standard Class",
    "Second Class",
    "First Class",
    "Same Day"
]

actions = []

for region in candidate_regions["Order Region"].unique():
    for mode in shipping_modes:
        actions.append(f"{region}|{mode}")

from sklearn.preprocessing import LabelEncoder

action_encoder = LabelEncoder()

action_ids = action_encoder.fit_transform(actions)

action_mapping = pd.DataFrame({
    "action": actions,
    "action_id": action_ids
})

action_mapping.to_csv(
    "action_mapping.csv",
    index=False
)

In [54]:
action = (
    candidate_regions,
    shipping_modes
)

In [55]:
action_encoder = LabelEncoder()

# Reconstruct 'Shipping Mode' string from 'shipping_mode_speed'
shipping_mode_map_reverse = {
    1: 'Standard Class',
    2: 'Second Class',
    3: 'First Class',
    4: 'Same Day'
}
train_df['Shipping Mode_str'] = train_df['shipping_mode_speed'].map(shipping_mode_map_reverse)

# Reconstruct 'Order Region' string from one-hot encoded columns
order_region_cols = [col for col in train_df.columns if col.startswith('Order Region_')]
# Use idxmax to find the column with value 1, then strip the prefix
train_df['Order Region_str'] = train_df[order_region_cols].idxmax(axis=1).str.replace('Order Region_', '')

# Create the 'action' column by combining the reconstructed strings
train_df['action'] = train_df['Order Region_str'] + '|' + train_df['Shipping Mode_str']

# Now fit and transform the action column
train_df['action_id'] = action_encoder.fit_transform(
    train_df['action']
)

joblib.dump(
    action_encoder,
    "action_encoder.pkl"
)

print(
    "Unique actions:",
    train_df['action_id'].nunique()
)

Unique actions: 4


# Check State Space Size

In [56]:
print(
    "Unique states:",
    train_df['state'].nunique()
)

print(
    "Unique actions:",
    train_df['action'].nunique()
)

print(
    "Q-table size:",
    train_df['state'].nunique()
    *
    train_df['action'].nunique()
)

Unique states: 1650
Unique actions: 4
Q-table size: 6600


# **STEP 6: Q-LEARNING IMPLEMENTATION**


In [57]:
train_df = pd.read_csv("dataco_rl_train_discrete.csv")

# Rebuild state strings if not already saved
state_columns = [
    'Product Category Id',
    'Category Name',
    'shipping_mode_speed',
    'stocking_score_bin',
    'margin_pct_bin',
    'shipping_delay_bin',
    'profit_per_item_bin',
    'unique_products_bin'
]

train_df['state'] = (
    train_df[state_columns]
    .astype(str)
    .agg('|'.join, axis=1)
)

# ------------------------------------------
# Rebuild actions
# ------------------------------------------

shipping_mode_map_reverse = {
    1: 'Standard Class',
    2: 'Second Class',
    3: 'First Class',
    4: 'Same Day'
}

train_df['Shipping Mode_str'] = (
    train_df['shipping_mode_speed']
    .map(shipping_mode_map_reverse)
)

region_cols = [
    c for c in train_df.columns
    if c.startswith("Order Region_")
]

train_df['Order Region_str'] = (
    train_df[region_cols]
    .idxmax(axis=1)
    .str.replace("Order Region_", "")
)

train_df['action'] = (
    train_df['Order Region_str']
    + "|"
    + train_df['Shipping Mode_str']
)

# ------------------------------------------
# Encode states/actions
# ------------------------------------------

from sklearn.preprocessing import LabelEncoder

state_encoder = LabelEncoder()
action_encoder = LabelEncoder()

train_df['state_id'] = state_encoder.fit_transform(
    train_df['state']
)

train_df['action_id'] = action_encoder.fit_transform(
    train_df['action']
)

# ==========================================
# Q-Learning Parameters
# ==========================================

alpha = 0.10      # learning rate
gamma = 0.95      # discount factor
epsilon = 0.20    # exploration rate

episodes = 50

n_states = train_df['state_id'].nunique()
n_actions = train_df['action_id'].nunique()

Q = np.zeros((n_states, n_actions))

# ==========================================
# Q-Learning Training
# ==========================================

episode_rewards = []

for episode in range(episodes):

    total_reward = 0

    for i in range(len(train_df)-1):

        state = train_df.iloc[i]['state_id']
        action = train_df.iloc[i]['action_id']
        reward = train_df.iloc[i]['reward']

        next_state = train_df.iloc[i+1]['state_id']

        # Q-learning update
        best_future_q = np.max(Q[next_state])

        Q[state, action] = (
            Q[state, action]
            + alpha * (
                reward
                + gamma * best_future_q
                - Q[state, action]
            )
        )

        total_reward += reward

    episode_rewards.append(total_reward)

    if episode % 10 == 0:
        print(
            f"Episode {episode}: "
            f"Reward = {total_reward:.2f}"
        )

print("Q-learning training complete")

Episode 0: Reward = -37344.69
Episode 10: Reward = -37344.69
Episode 20: Reward = -37344.69
Episode 30: Reward = -37344.69
Episode 40: Reward = -37344.69
Q-learning training complete


# Extract the Learned Policy

In [58]:
policy = {}

for state in range(n_states):

    best_action = np.argmax(Q[state])

    policy[state] = best_action

print("Policy size:", len(policy))

Policy size: 1650


# Convert Best Actions Back to Business Decisions

In [59]:
best_actions = []

for state in range(n_states):

    action_id = np.argmax(Q[state])

    action_name = action_encoder.inverse_transform(
        [action_id]
    )[0]

    best_actions.append(
        [state, action_name]
    )

policy_df = pd.DataFrame(
    best_actions,
    columns=["state_id", "recommended_action"]
)

policy_df.head()

,state_id,recommended_action
0,0,West Africa|First Class
1,1,West Africa|First Class
2,2,West Africa|Same Day
3,3,West Africa|Same Day
4,4,West Africa|Same Day


# Save Results

In [ ]:
policy_df.to_csv(
    "q_learning_policy.csv",
    index=False
)

np.save(
    "q_table.npy",
    Q
)

print("Policy and Q-table saved")

# **STEP 7: SARSA IMPLEMENTATION**


# **STEP 8: COMPARISON OF PROFIT, REWARD, AND ON-TIME DELIVERY PERFORMANCE**


* State = information known BEFORE shipment decision
* Action = shipping method / route / priority decision
* Reward = profit − lateness penalty
* Next state = next order environment

Reward Function

Rt=α(Profit)−β(Late Shipment Penalty)

Where:

α controls profit importance

β controls delivery performance importance

A more detailed version:

Rt=αPt−βLt−γCt

Where:
Pt = profit

Lt = lateness indicator or delay days

Ct = shipping cost

# STEP 8: Q-learning or SARSA environment setup